# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import trim, col

# read Bronze table

In [0]:
df = spark.table("workspace.bronze.erp_loc_a101")

df.display()

In [0]:
# Fix Cntry names
# Fix CID removing - 
# Fix trimming columns

# Silver Transformations

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## CustomerId Cleanup

In [0]:
df = df.withColumn("cid", F.regexp_replace(col("CID"), "-", ""))

## Country Normalization

In [0]:
df = df.withColumn(
    "cntry",
    F.when(col('CNTRY') == "DE", "Germany")
     .when(col("CNTRY").isin("US", "USA"), "United States")
     .when((col("CNTRY") == "") | col("CNTRY").isNull(), "n/a")
     .otherwise(col("CNTRY"))
)

## Renaming Columns

In [0]:
RENAME_MAP = {
    "cid" : "customer_number",
    "cntry" : "country"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.limit(10).display()

# Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customer_location")

In [0]:
%sql
select * from workspace.silver.erp_customers limit 10